# Initial Data Treatment

In [1]:
import os
import re
import unicodedata
from datetime import datetime
import pandas as pd
import polars as pl
from lingua import LanguageDetectorBuilder, Language

# ingestion path
CHANNELS_PATH = '../data/raw/channels_info.csv'
VIDEOS_PATH = '../data/raw/videos_info.csv'
COMMENTS_PATH = '../data/raw/comments_info.csv'
KEY_TERMS_PATH = '../data/cleaned/key_terms.txt'
EXCLUDE_TERMS_PATH = '../data/cleaned/exclude_terms.txt'
CORE_TERMS_PATH = '../data/cleaned/core_terms.txt'
DIFF_TERMS_PATH = '../data/cleaned/diff_terms.txt'

# saving path
CHANNELS_CLEANED_PATH = '../data/cleaned/cleaned_channels_info.csv'
VIDEOS_CLEANED_PATH = '../data/cleaned/cleaned_videos_info.csv'
COMMENTS_CLEANED_PATH = '../data/cleaned/cleaned_comments_info.csv'

# exclusive upper bound: keeps records through 2025-12-31
CUTOFF_DATE = datetime(2026, 1, 1)

# creates cleaned folder in case it doesn't exists
os.makedirs('../data/cleaned', exist_ok=True)

# relevant channels info
COLS_CHANNELS = ['channel_id', 'title', 'description', 'published_at', 'country', 'view_count', 'comment_count', 'subscriber_count', 'video_count', 'keywords', 'profile_picture_url']

# relevant videos info
COLS_VIDEOS = ['video_id', 'title', 'description', 'channel_id', 'published_at', 'category_id', 'tags', 'view_count', 'like_count', 'comment_count', 'duration', 'licensed_content', 'is_made_for_kids', 'thumbnail_url', 'default_audio_language', 'default_language', 'topicCategories', 'is_short']

# relevant comments info
COLS_COMMENTS = ['video_id', 'comment_id', 'author_profile_image_url', 'author_channel_url', 'author_channel_id', 'comment', 'published_at', 'like_count', 'is_reply', 'parent_id', 'channel_id']

# model for language detection
detector = LanguageDetectorBuilder.from_all_languages().build()

In [2]:
def normalize_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    normalized_text = unicodedata.normalize('NFKD', text)
    return "".join([c for c in normalized_text if not unicodedata.combining(c)])

def load_terms(terms_path):
    TERMS = []

    with open(terms_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            
            if not line or line.startswith("#"):
                continue
                
            parts = [p.strip() for p in line.split(",") if p.strip()]
            
            for term in parts:
                term_limpo = normalize_text(term.lower())
                TERMS.append(term_limpo)
    
    return TERMS

KEY_TERMS = load_terms(KEY_TERMS_PATH)
EXCLUDE_TERMS = load_terms(EXCLUDE_TERMS_PATH)
CORE_TERMS = load_terms(CORE_TERMS_PATH)
DIFF_TERMS = load_terms(DIFF_TERMS_PATH)

In [3]:
print(KEY_TERMS)
print(EXCLUDE_TERMS)
print(CORE_TERMS)
print(DIFF_TERMS)

['fluoxetina', 'prozac', 'daforin', 'fluxene', 'verotina', 'sertralina', 'zoloft', 'tolrest', 'afetus', 'dieloft', 'paroxetina', 'paxil', 'pondera', 'roxetin', 'paxtrat', 'cebrilin', 'citalopram', 'cipramil', 'procimax', 'citta', 'alcytam', 'maxapram', 'denyl', 'escitalopram', 'lexapro', 'esc odt', 'reconter', 'espran', 'fluvoxamina', 'luvox', 'revoc', 'venlafaxina', 'efexor xr', 'venlaxin', 'venlifit od', 'vensate', 'desvenlafaxina', 'pristiq', 'elifore', 'desve', 'zodel', 'imense', 'deller', 'desduo', 'duloxetina', 'cymbalta', 'velija', 'cymbi', 'leduo', 'amitriptilina', 'tryptanol', 'amytril', 'clomipramina', 'anafranil', 'imipramina', 'tofranil', 'nortriptilina', 'pamelor', 'nortril', 'maprotilina', 'ludiomil', 'doxepina', 'aponal', 'sinequan', 'mirtazapina', 'remeron', 'remeron soltab', 'menelat', 'razapina', 'mianserina', 'tolvon', 'maprotilina', 'ludiomil', 'menelat odt', 'razapina odt', 'tranilcipromina', 'parnate', 'sulpan', 'moclobemida', 'aurorix', 'selegilina', 'eldepryl', 

In [4]:
# uses lingua for language detection, known for working well even with few words
def is_portuguese(comment: str) -> bool:
    lang = detector.detect_language_of(str(comment).strip())

    return lang == Language.PORTUGUESE

## Channels

In [5]:
df_channels = pd.read_csv(CHANNELS_PATH, usecols=COLS_CHANNELS)
df_channels.head()

,channel_id,title,description,published_at,country,view_count,comment_count,subscriber_count,video_count,keywords,profile_picture_url
0,UCJ3WGzxEBfttrG9PC3yGuGA,Dra Juliana Bragalda,"Instagram: @drajulianabragalda\nPsiquiatria, s...",2016-02-04T17:04:36Z,BR,42929.0,0.0,131.0,26.0,medicina saude psiquiatria remedio,https://yt3.ggpht.com/rmjLYbnw1opym4NyUeYNLHb3...
1,UCC1_Drela_P4MAW8pPcdjSw,Julio Pereira - Neurocirurgião,O maior Canal de Neurocirurgia do mundo. Vincu...,2011-03-27T23:41:33Z,BR,238109410.0,0.0,1170000.0,13305.0,"Neurocirurgia neurocirurgiabr ""julio pereira n...",https://yt3.ggpht.com/ytc/AIdro_kCExGHXRLOsOeS...
2,UCC1_Drela_P4MAW8pPcdjSw,Julio Pereira - Neurocirurgião,O maior Canal de Neurocirurgia do mundo. Vincu...,2011-03-27T23:41:33Z,BR,238109410.0,0.0,1170000.0,13305.0,"Neurocirurgia neurocirurgiabr ""julio pereira n...",https://yt3.ggpht.com/ytc/AIdro_kCExGHXRLOsOeS...
3,UCwxbzB79I8_J6VgkAr_9AmQ,Dr. Francisco Zacarias,Transformador de vidas e colecionador de histó...,2012-03-18T19:01:26Z,BR,44682552.0,0.0,717000.0,1728.0,"""Dr. Francisco Zacarias"" ""Francisco Zacarias"" ...",https://yt3.ggpht.com/UwPcAZwQN4DWNX4zBDWiKh2P...
4,UCd2YBETxQL8YvNZU7rOjvCw,Dr. Guilherme Labinas,Dr. Labinas é médico psiquiatra da Unicamp e c...,2016-03-02T15:44:50Z,BR,15367347.0,0.0,128000.0,1472.0,psiquiatria; depressao ansiedade #institutolab...,https://yt3.ggpht.com/ytc/AIdro_nGo4A1CJmA3HVI...


In [6]:
# passing to datetime and ordering
df_channels['published_at'] = pd.to_datetime(df_channels['published_at'], utc=True, format='mixed')
df_channels['published_at'] = df_channels['published_at'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
df_channels.sort_values(by='published_at', inplace=True)

In [7]:
df_channels.head()

,channel_id,title,description,published_at,country,view_count,comment_count,subscriber_count,video_count,keywords,profile_picture_url
163154,UCvC4D8onUfXzvjTOM-dBfEA,Marvel Entertainment,Welcome to Marvel’s official YouTube channel! ...,2005-06-16 09:09:27,US,7.218430e+09,0.0,22700000.0,10433.0,"Marvel Comics ""Comic Books"" Spiderman Spider-M...",https://yt3.ggpht.com/k7BhK-hm9_MbJbaKznHPhir6...
161520,UC238kiEEktbDUpIR7jMBz1w,Eurovision Song Contest | Deutschland,Der offizielle YouTube-Channel der ARD zum Eur...,2005-08-11 01:31:26,DE,6.555603e+07,0.0,67600.0,1385.0,"""Eurovision Song Contest"" ESC ""Grand Prix d'Eu...",https://yt3.ggpht.com/ZYw1zK45CSDzkUReqBeaIASI...
159719,UC238kiEEktbDUpIR7jMBz1w,Eurovision Song Contest | Deutschland,Der offizielle YouTube-Channel der ARD zum Eur...,2005-08-11 01:31:26,DE,6.554458e+07,0.0,67600.0,1385.0,"""Eurovision Song Contest"" ESC ""Grand Prix d'Eu...",https://yt3.ggpht.com/ZYw1zK45CSDzkUReqBeaIASI...
161524,UC238kiEEktbDUpIR7jMBz1w,Eurovision Song Contest | Deutschland,Der offizielle YouTube-Channel der ARD zum Eur...,2005-08-11 01:31:26,DE,6.555603e+07,0.0,67600.0,1385.0,"""Eurovision Song Contest"" ESC ""Grand Prix d'Eu...",https://yt3.ggpht.com/ZYw1zK45CSDzkUReqBeaIASI...
277,UC238kiEEktbDUpIR7jMBz1w,Eurovision Song Contest | Deutschland,Der offizielle YouTube-Channel der ARD zum Eur...,2005-08-11 01:31:26,DE,6.053526e+07,0.0,65100.0,1341.0,"""Eurovision Song Contest"" ESC ""Grand Prix d'Eu...",https://yt3.ggpht.com/ZYw1zK45CSDzkUReqBeaIASI...


In [8]:
df_channels.isna().sum()

channel_id                14
title                     23
description            26662
published_at              14
country                59264
view_count                14
comment_count             14
subscriber_count          14
video_count               14
keywords               74618
profile_picture_url       14
dtype: int64

In [9]:
# saving number of channels for later comparison
initial_channels_number = df_channels['channel_id'].nunique()

# filter only channels relevant to brazilian context
df_channels = df_channels[(df_channels['country'] == 'BR') | (df_channels['country'].isnull())]
df_channels.isna().sum()

channel_id                14
title                     18
description            21372
published_at              14
country                59264
view_count                14
comment_count             14
subscriber_count          14
video_count               14
keywords               57286
profile_picture_url       14
dtype: int64

In [10]:
# keep the first record after chronological ordering (oldest publication)
df_channels.drop_duplicates(subset=['channel_id'], keep='first', inplace=True)
df_channels.duplicated().sum()

np.int64(0)

In [11]:
# fill missing values for description and keywords using a empty string
# and droping any missing data that remains
df_channels.fillna(
    {
        'description': '',
        'keywords': '',
        'country': 'Unknown'
    },
    inplace=True
)
df_channels.dropna(inplace=True)

# ids for further filtering
channels_ids = df_channels['channel_id'].tolist()

In [12]:
df_channels.drop(columns=['country'], inplace=True)

In [13]:
print(f"After the initial treatment, {df_channels['channel_id'].nunique()} channels remained, out of {initial_channels_number}")

After the initial treatment, 60031 channels remained, out of 103838


In [14]:
print(f'DataFrame containing channels information was saved at {CHANNELS_CLEANED_PATH}')
df_channels.to_csv(CHANNELS_CLEANED_PATH, index=False, encoding='utf-8')

DataFrame containing channels information was saved at ../data/cleaned/cleaned_channels_info.csv


## Videos

In [15]:
df_videos = pd.read_csv(VIDEOS_PATH, usecols=COLS_VIDEOS)
df_videos.head()

,video_id,title,description,channel_id,published_at,category_id,tags,view_count,like_count,comment_count,duration,licensed_content,is_made_for_kids,thumbnail_url,default_audio_language,default_language,topicCategories,is_short
0,ENE2DM1s4L0,Tudo sobre a Fluoxetina #psiquiatria #ansiedade,Tudo sobre o Fluoxetina \n#psiquiatria #ansiedade,UCJ3WGzxEBfttrG9PC3yGuGA,2026-03-30T20:44:08Z,22,"['Psiquiatria', 'Psiquiatra', 'Fluoxetina', 'A...",852,31,3,PT1M25S,False,False,https://i.ytimg.com/vi/ENE2DM1s4L0/hqdefault.jpg,pt-BR,pt-PT,"['https://en.wikipedia.org/wiki/Health', 'http...",True
1,SNYj_s4NgJo,✅ Qual vence? Fluoxetina ou Sertralina para de...,Mantenha-se informado e proteja sua saúde com ...,UCC1_Drela_P4MAW8pPcdjSw,2026-04-08T12:53:39Z,27,"['neurocirurgiabr', 'neurocirurgia', 'julio pe...",10752,45,2,PT2M59S,True,False,https://i.ytimg.com/vi/SNYj_s4NgJo/hqdefault.jpg,pt,pt-BR,"['https://en.wikipedia.org/wiki/Health', 'http...",True
2,e1I0geRSlSA,Sertralina x Fluoxetina: para que serve CADA u...,Mantenha-se informado e proteja sua saúde com ...,UCC1_Drela_P4MAW8pPcdjSw,2026-04-04T18:42:38Z,27,"['neurocirurgiabr', 'neurocirurgia', 'julio pe...",3992,22,1,PT2M58S,True,False,https://i.ytimg.com/vi/e1I0geRSlSA/hqdefault.jpg,pt,pt-BR,"['https://en.wikipedia.org/wiki/Health', 'http...",True
3,emFwxm8i4Iw,Fluoxetina: a “pílula da felicidade”,NaN,UCwxbzB79I8_J6VgkAr_9AmQ,2026-04-20T19:53:53Z,22,[],5323,105,1,PT1M,True,False,https://i.ytimg.com/vi/emFwxm8i4Iw/hqdefault.jpg,pt-BR,pt-PT,['https://en.wikipedia.org/wiki/Health'],True
4,SK_FBULwoA8,ANTIDEPRESSIVO: POR QUE PIORA ANTES DE MELHORA...,Você começou a tomar um antidepressivo para me...,UCd2YBETxQL8YvNZU7rOjvCw,2026-04-10T23:00:35Z,27,"['#institutolabinas', '#drguilhermelabinas', '...",3769,119,13,PT45S,True,False,https://i.ytimg.com/vi/SK_FBULwoA8/hqdefault.jpg,pt-BR,pt-BR,"['https://en.wikipedia.org/wiki/Health', 'http...",True


In [16]:
df_videos.shape

(190192, 18)

In [17]:
# passing to datetime and ordering
df_videos['published_at'] = pd.to_datetime(df_videos['published_at'], utc=True, format='mixed')
df_videos['published_at'] = df_videos['published_at'].dt.tz_convert('America/Sao_Paulo').dt.tz_localize(None)
df_videos.sort_values(by='published_at', inplace=True)

In [18]:
# keep the first record after chronological ordering (oldest publication)
df_videos.drop_duplicates(subset=['video_id'], keep='first', inplace=True)
df_videos.duplicated().sum()

np.int64(0)

In [19]:
df_videos.isna().sum()

video_id                      0
title                         0
description               34947
channel_id                    0
published_at                  0
category_id                   0
tags                          0
view_count                    0
like_count                    0
comment_count                 0
duration                     10
licensed_content              0
is_made_for_kids              0
thumbnail_url                 0
default_audio_language    45632
default_language              1
topicCategories               0
is_short                      0
dtype: int64

In [20]:
initial_videos_number = df_videos['video_id'].nunique()

# fill missing description with a empty string and drop missing values
df_videos.fillna({'description': ''}, inplace=True)
df_videos = df_videos.dropna(subset=['video_id', 'title', 'channel_id', 'published_at', 'duration'])

In [21]:
# keeping only videos through the end of 2025, for plotting purposes
df_videos = df_videos[df_videos['published_at'] < CUTOFF_DATE]
df_videos['published_at'].max(), df_videos['published_at'].min()

(Timestamp('2025-12-31 23:45:06'), Timestamp('2015-12-31 22:57:23'))

In [22]:
# bool cols for filtering portuguese content
df_videos['audio_is_pt'] = df_videos['default_audio_language'].str.startswith('pt', na=False)
df_videos['language_is_pt'] = df_videos['default_language'].str.startswith('pt', na=False)

# filtering based on channel region
df_videos = df_videos[df_videos['channel_id'].isin(channels_ids)]

# filtering based on video language
df_videos = df_videos[(df_videos['audio_is_pt']) | (df_videos['language_is_pt'])]

In [23]:
# filters videos where the title description:
# contain one of the key terms
# contain one of the diff terms with a core term
# does not contain any exclude terms
pattern_pos = r'\b(?:' + '|'.join(re.escape(term) for term in KEY_TERMS) + r')\b'
pattern_neg = r'\b(?:' + '|'.join(re.escape(term) for term in EXCLUDE_TERMS) + r')\b'
pattern_cor = r'\b(?:' + '|'.join(re.escape(term) for term in CORE_TERMS) + r')\b'
pattern_dif = r'\b(?:' + '|'.join(re.escape(term) for term in DIFF_TERMS) + r')\b'

df_videos['title_description'] = df_videos['title'] + ' ' + df_videos['description']
df_videos['title_description'] = df_videos['title_description'].apply(lambda x: normalize_text(x.lower()))

has_key     = df_videos['title_description'].str.contains(pattern_pos, case=False, na=False)
has_diff    = df_videos['title_description'].str.contains(pattern_dif, case=False, na=False)
has_core    = df_videos['title_description'].str.contains(pattern_cor, case=False, na=False)
has_exclude = df_videos['title_description'].str.contains(pattern_neg, case=False, na=False)

df_videos = df_videos[
    (has_key | (has_diff & has_core)) & (~has_exclude)
] 

# ids for further filtering
videos_ids = df_videos['video_id'].tolist()

In [24]:
df_videos.drop(columns=['audio_is_pt', 'language_is_pt', 'default_audio_language', 'default_language', 'title_description'], inplace=True)

In [25]:
print(f"After the initial treatment, {df_videos['video_id'].nunique()} videos remained out of {initial_videos_number}")

After the initial treatment, 17225 videos remained out of 190156


In [26]:
print(f'DataFrame containing videos information was saved at {VIDEOS_CLEANED_PATH}')
df_videos.to_csv(VIDEOS_CLEANED_PATH, index=False, encoding='utf-8')

DataFrame containing videos information was saved at ../data/cleaned/cleaned_videos_info.csv


## Comments

In [27]:
# using polasr to read comments of videos filtered in aforementioned stage
df_comments = (
    pl.scan_csv(COMMENTS_PATH, try_parse_dates=True)
    .select(COLS_COMMENTS)
    .filter(pl.col('video_id').is_in(videos_ids))
    .collect()
)

df_comments.head()

video_id,comment_id,author_profile_image_url,author_channel_url,author_channel_id,comment,published_at,like_count,is_reply,parent_id,channel_id
str,str,str,str,str,str,"datetime[μs, UTC]",i64,bool,str,str
"""C90bUPpuSZ0""","""UgzjYw4d6UViuoumZfJ4AaABAg""","""https://yt3.ggpht.com/ytc/AIdr…","""http://www.youtube.com/@tiabab…","""UCb5BvXTNZl2cksRjKpfAZLw""","""Pergunto porque creio que isso…",2026-03-04 20:58:17 UTC,0,false,null,"""UCbY3R6CQ150IU3ovyZaxU-A"""
"""C90bUPpuSZ0""","""UgzjYw4d6UViuoumZfJ4AaABAg.ATx…","""https://yt3.ggpht.com/ytc/AIdr…","""http://www.youtube.com/@leandr…","""UCi0vLmjMhbTKWhlmY31kdqg""","""Seu filho ainda está tomando?""",2026-04-18 00:01:31 UTC,0,true,"""UgzjYw4d6UViuoumZfJ4AaABAg""","""UCbY3R6CQ150IU3ovyZaxU-A"""
"""C90bUPpuSZ0""","""UgyTD0Miw_UgKyUqIux4AaABAg""","""https://yt3.ggpht.com/ytc/AIdr…","""http://www.youtube.com/@tiabab…","""UCb5BvXTNZl2cksRjKpfAZLw""","""Porque é um erro começar com d…",2026-03-04 20:52:29 UTC,0,false,null,"""UCbY3R6CQ150IU3ovyZaxU-A"""
"""TT7bNmwxAj4""","""Ugx0fBHpru2EHTxwGz14AaABAg""","""https://yt3.ggpht.com/ytc/AIdr…","""http://www.youtube.com/@demose…","""UCgrk33SH8v2Ij4y3hCcSUKQ""","""E em homens como são os efeito…",2025-12-31 01:17:19 UTC,0,false,null,"""UCzJ4TDRp2N822xCoOn6DWdQ"""
"""p7TZuBnvfag""","""UgyUI-Xr-_PDlVvvHoh4AaABAg""","""https://yt3.ggpht.com/ytc/AIdr…","""http://www.youtube.com/@regina…","""UC3d8VE02-ywldepQJOU8UAg""","""Boa tarde Dor eu tô tomando qu…",2025-12-30 18:57:06 UTC,0,false,null,"""UCOX78J0UD_rVssU8x_dKLTg"""


In [28]:
initial_comments_number = (
    pl.scan_csv(COMMENTS_PATH)
    .select(pl.col('comment_id').n_unique())
    .collect()
    .item()
)

initial_users_number = (
    pl.scan_csv(COMMENTS_PATH)
    .select(pl.col('author_channel_id').n_unique())
    .collect()
    .item()
)

In [29]:
# converting timezone
df_comments = df_comments.with_columns(
    pl.col('published_at')
    .dt.convert_time_zone('America/Sao_Paulo')
    .dt.replace_time_zone(None)
)

In [30]:
df_comments = (
    df_comments
    # preserve a deterministic first occurrence from the input order
    .unique(subset=['comment_id'], keep='first', maintain_order=True)
    # filters for comments that contain text and min length of 3
    .filter(
        (pl.col('published_at') < CUTOFF_DATE) &
        (pl.col('comment').is_not_null()) &
        (pl.col('comment').str.count_matches(r'[A-Za-zÀ-ÿ]') >= 3)
    )
    # filters for comments that language is portuguese
    .filter(
        pl.col('comment').map_elements(is_portuguese, return_dtype=pl.Boolean)
    )
)

In [31]:
print(f"After the initial treatment, {df_comments['comment_id'].n_unique()} comments remained out of {initial_comments_number}")
print(f"After the initial treatment, {df_comments['author_channel_id'].n_unique()} unique authors remained out of {initial_users_number}")

After the initial treatment, 1135946 comments remained out of 35572568
After the initial treatment, 499683 unique authors remained out of 19611547


In [32]:
print('Final values for database:')
print(f'Number of channels: {df_comments["channel_id"].n_unique()}')
print(f'Number of videos: {df_comments["video_id"].n_unique()}')
print(f'Number of comments: {df_comments["comment_id"].n_unique()}')
print(f'Number of unique authors: {df_comments["author_channel_id"].n_unique()}')

Final values for database:
Number of channels: 7382
Number of videos: 15808
Number of comments: 1135946
Number of unique authors: 499683


In [33]:
print(f'Percentage of channels kept: {df_comments["channel_id"].n_unique() / initial_channels_number * 100:.2f}%')
print(f'Percentage of videos kept: {df_comments["video_id"].n_unique() / initial_videos_number * 100:.2f}%')
print(f'Percentage of comments kept: {df_comments["comment_id"].n_unique() / initial_comments_number * 100:.2f}%')
print(f'Percentage of unique authors kept: {df_comments["author_channel_id"].n_unique() / initial_users_number * 100:.2f}%')

Percentage of channels kept: 7.11%
Percentage of videos kept: 8.31%
Percentage of comments kept: 3.19%
Percentage of unique authors kept: 2.55%


In [34]:
print(f'DataFrame containing comments information was saved at {COMMENTS_CLEANED_PATH}')
df_comments.write_csv(COMMENTS_CLEANED_PATH)

DataFrame containing comments information was saved at ../data/cleaned/cleaned_comments_info.csv
